## ADC Linker LSTM Model

In [1]:
# following this tutorial on LSTMs in pytorch: https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-networks-using-pytorch/

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.transforms as TF
import numpy as np
import pandas as pd
import re
import math

In [2]:
df = pd.read_pickle("data/adc_data_complete_v2.pkl")

In [3]:
df['smiles'].isnull().any()

np.False_

In [4]:
df.head()

,index_x,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,Product name,smiles,calc_SA_score,TPSA,QED,LogP,...,amino,phosphate,total_rings,aromatic_rings,aliphatic_rings,num_h_donors,num_h_acceptors,FractionCSP3,num_rot_bonds,Surface Area
0,0,0,0,0,(Ac)Phe-Lys(Alloc)-PABC-PNP,C=CCOC(=O)NCCCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC(...,3.508295,204.30,0.036588,4.5636,...,0,0,3,3,0,4,10,0.285714,18,288.060675
1,2,2,2,2,Fmoc-Val-Cit-PAB,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.338596,171.88,0.163403,3.6140,...,0,0,4,3,1,6,6,0.333333,13,256.107099
2,3,3,3,3,Fmoc-Val-Cit-PAB-PNP,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.753739,230.32,0.030495,5.7455,...,0,0,5,4,1,5,10,0.275000,16,321.776490
3,4,4,4,4,Mc-Val-Cit-PABC-PNP,CC(C)[C@H](NC(=O)CCCCCN1C(=O)C=CC1=O)C(=O)N[C@...,3.796268,258.47,0.032958,2.8084,...,0,0,3,2,1,5,11,0.400000,20,304.496626
4,5,5,5,5,Val-cit-PAB-OH,CC(C)[C@H](N)C(=O)N[C@@H](CCCNC(N)=O)C(=O)Nc1c...,2.890504,159.57,0.315935,0.0340,...,1,0,1,1,0,6,5,0.500000,10,158.418876


In [5]:
# SMILES tokenizer reference: https://deepchem.readthedocs.io/en/2.4.0/api_reference/tokenizers.html

SMI_REGEX = re.compile(r"(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|B|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\|\/|:|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])")
smiles_tokens = df['smiles'].str.findall(SMI_REGEX)

# smiles tokens is a series of lists for all patterns found per smiles string
# explode converts into a single series with all pattern matches
unique_tokens = smiles_tokens.explode().unique()
smiles_map = {"PAD": 0}
smiles_map_rest = {char: i+1 for i,char in enumerate(unique_tokens)}
smiles_map.update(smiles_map_rest)
#smiles_map

In [6]:
# convert smiles to their integer representation example

smiles_ex = 'CC(=O)N[C@@H](CC1=CC=CC=C1)C(=O)N[C@@H](CCCCNC(=O)OCC=C)C(=O)NC2=CC=C(C=C2)COC(=O)OC3=CC=C(C=C3)[N+](=O)[O-]'
smiles_char_tokens = SMI_REGEX.findall(smiles_ex)
smiles_token_seq = [smiles_map[char] for char in smiles_char_tokens]
#smiles_char_seq

In [ ]:
# test set up for embedding layer for pytorch to handles these tokens

embedding_dim = 16 # this is num dimensions that LSTM will use to describe each smiles character

embedding_layer = nn.Embedding(num_embeddings=len(unique_tokens), embedding_dim=embedding_dim, padding_idx=smiles_map["PAD"])
embedding_layer.weight.shape #30 unique smiles patterns by 16 dimensions per pattern

torch.Size([30, 16])

In [ ]:
# test embedding on our smiles example string

input_tensor = torch.LongTensor(smiles_token_seq) # this is just our token representation converted to tensor
embedded_seq = embedding_layer(input_tensor) # now each token is represented by three dimension vector of random initialized weights
#embedded_seq

In [9]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.embedding = nn.Embedding(
            num_embeddings=len(smiles_map),
            embedding_dim=input_dim,
            padding_idx=smiles_map["PAD"],
        )
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded_seq = self.embedding(x)
        out, (hn, cn) = self.lstm(embedded_seq)
        out = self.fc(out[:, -1, :])  # Take last time step
        return out

In [10]:
model = LSTMModel(input_dim=16, hidden_dim=100, layer_dim=5, output_dim=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Training on device: {device}")

Training on device: cuda


In [23]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [24]:
def smiles_to_tokens(smile_pattern_list):
    return [smiles_map[char] for char in smile_pattern_list]


def collate_smiles(batch):
    # this adds the padding, short smiles will get padded to length of longest smile in batch
    smiles = [item[0] for item in batch]
    scores = [item[1] for item in batch]

    smiles_padded = pad_sequence(
        smiles, batch_first=True, padding_value=smiles_map["PAD"]
    )

    scores_stack = torch.stack(
        scores
    )  # scores is a list from batch, stack reshapes to a size of (batch,1)

    return smiles_padded, scores_stack


class adcDataset(Dataset):
    def __init__(self, directory):
        self.directory = directory
        self.file = pd.read_pickle(directory)
        self.smiles = df["smiles"].str.findall(SMI_REGEX).apply(smiles_to_tokens)
        self.SA_score = torch.tensor(np.array((self.file["calc_SA_score"])))

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = torch.tensor(self.smiles[idx])
        score = torch.tensor([self.SA_score[idx]])

        return smile, score

In [25]:
dataset = adcDataset("data/adc_data_complete_v2.pkl")
loader = DataLoader(
    dataset,
    batch_size=1000,
    shuffle=True,
    collate_fn=collate_smiles,
)


In [27]:
num_epochs = 20

for epoch in range(num_epochs):
    for i, (smile, score) in enumerate(loader):
        print(f"Batch {i+1}/{math.ceil(len(dataset)/loader.batch_size)}", end="\r")
        smile, label = smile.to(device), score.to(device)

        optimizer.zero_grad()

        output = model(smile)  # forward
        loss = criterion(output.float(), label.float())  # calculate cross entropy loss
        loss.backward()  # back propagation
        optimizer.step()

    print()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.3f}")

Batch 3/3
Epoch 1/20, Loss: 0.073
Batch 3/3
Epoch 2/20, Loss: 0.064
Batch 3/3
Epoch 3/20, Loss: 0.057
Batch 3/3
Epoch 4/20, Loss: 0.077
Batch 3/3
Epoch 5/20, Loss: 0.055
Batch 3/3
Epoch 6/20, Loss: 0.087
Batch 3/3
Epoch 7/20, Loss: 0.060
Batch 3/3
Epoch 8/20, Loss: 0.081
Batch 3/3
Epoch 9/20, Loss: 0.055
Batch 3/3
Epoch 10/20, Loss: 0.103
Batch 3/3
Epoch 11/20, Loss: 0.057
Batch 3/3
Epoch 12/20, Loss: 0.066
Batch 3/3
Epoch 13/20, Loss: 0.167
Batch 3/3
Epoch 14/20, Loss: 0.066
Batch 3/3
Epoch 15/20, Loss: 0.072
Batch 3/3
Epoch 16/20, Loss: 0.138
Batch 3/3
Epoch 17/20, Loss: 0.069
Batch 3/3
Epoch 18/20, Loss: 0.058
Batch 3/3
Epoch 19/20, Loss: 0.058
Batch 3/3
Epoch 20/20, Loss: 0.074


In [43]:
SA_predictions = []
SA_true = []

with torch.no_grad():
    for i, (smile, score) in enumerate(loader):
        smile, label = smile.to(device), score.to(device)
        output = model(smile)
        SA_predictions += output
        SA_true += score

# convert from tensor to array for predicted and true scores
SA_predictions = np.array([pred.cpu().numpy() for pred in SA_predictions])
SA_true = np.array([true.cpu().numpy() for true in SA_true])

avg_percent_error = np.mean(np.abs((SA_true - SA_predictions) / (SA_true + 1e-8)) * 100)
print(f"Mean Absolute Percent Error: {avg_percent_error:.2f}%")
        

Mean Absolute Percent Error: 6.18%
